# Importando as Bibliotecas

In [1]:
import pandas as pd
import re

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

print("Bibliotecas Importadas com Sucesso!!")

Bibliotecas Importadas com Sucesso!!


# Leitura da Base de Dados

In [2]:
dados = pd.read_csv("../data/dataset_tratado.csv", index_col=0)
dados.head()

,review_comment_message,review_score,sentimento
3,Recebi bem antes do prazo estipulado.,5,positivo
4,Parabéns lojas lannister adorei comprar pela I...,5,positivo
9,aparelho eficiente. no site a marca do aparelh...,4,positivo
12,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",4,positivo
15,"Vendedor confiável, produto ok e entrega antes...",5,positivo


# 1. Pré-Processamento

## Limpando o Texto

In [3]:
df_teste = dados.copy()

In [4]:
CONTRACTIONS = {
    'vc': 'você', 'tb': 'também', 'tbm': 'também', 'pq': 'por que', 'obg': 'obrigado', 'blz': 'beleza',
    'td': 'tudo', 'msm': 'mesmo', 'vlw': 'valeu', 'n': 'não', 'naum': 'não', 'cmprar': 'comprar',
    'mt': 'muito', 'pra': 'para'
}

In [5]:
def limpar_texto(texto):

    # 1. Lowercase
    texto = texto.lower()
    # 2. Remove HTML
    texto = re.sub(r'<[^>]+>', ' ', texto)
    # 3. Remove URLs
    texto = re.sub(r'http\S+|www\.\S+', ' ', texto)
    # 4. Remove e-mails
    texto = re.sub(r'\S+@\S+', ' ', texto)
    # 5. Expande contrações
    # Primeiro quebra o texto em palavras (texto.split())
    # Depois troca as contrações no dicionário (CONTRACTIONS)
    # Por fim junta tudo
    palavras = [CONTRACTIONS.get(p,p) for p in texto.split()]
    texto = " ".join(palavras)

    # 6. Remove caracteres indesejados
    texto = re.sub(r'[^a-záàâãéêèíìîóôõöúùûüç\s.,!?]', ' ', texto)

    # 7. Normaliza espaços
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto

In [7]:
df_teste["texto_limpo"] = df_teste["review_comment_message"].apply(limpar_texto)

df_teste[["review_comment_message", "texto_limpo"]].head()

,review_comment_message,texto_limpo
3,Recebi bem antes do prazo estipulado.,recebi bem antes do prazo estipulado.
4,Parabéns lojas lannister adorei comprar pela I...,parabéns lojas lannister adorei comprar pela i...
9,aparelho eficiente. no site a marca do aparelh...,aparelho eficiente. no site a marca do aparelh...
12,"Mas um pouco ,travando...pelo valor ta Boa.\r\n","mas um pouco ,travando...pelo valor ta boa."
15,"Vendedor confiável, produto ok e entrega antes...","vendedor confiável, produto ok e entrega antes..."


## Tokenização - Quebrando texto em Unidades

In [8]:
def tokenizacao(texto):

    return word_tokenize(texto, language="portuguese")

In [9]:
df_teste["texto_tokenizado"] = df_teste["texto_limpo"].apply(tokenizacao)

df_teste[["texto_limpo", "texto_tokenizado"]].head()

,texto_limpo,texto_tokenizado
3,recebi bem antes do prazo estipulado.,"[recebi, bem, antes, do, prazo, estipulado, .]"
4,parabéns lojas lannister adorei comprar pela i...,"[parabéns, lojas, lannister, adorei, comprar, ..."
9,aparelho eficiente. no site a marca do aparelh...,"[aparelho, eficiente, ., no, site, a, marca, d..."
12,"mas um pouco ,travando...pelo valor ta boa.","[mas, um, pouco, ,, travando, ..., pelo, valor..."
15,"vendedor confiável, produto ok e entrega antes...","[vendedor, confiável, ,, produto, ok, e, entre..."


## Stop Words - removendo as palavras frequentes

In [11]:
STOP_WORDS = set(stopwords.words("portuguese"))

NEGACOES = {"não", "nao", "nunca", "nem", "sem"}

STOP_WORDS_PT = STOP_WORDS - NEGACOES

In [13]:
def remove_stop_words(tokens):

    resultado = []
    for token in tokens:
        if (token.isalpha() and token not in STOP_WORDS_PT):
            resultado.append(token)
    
    return resultado

In [14]:
df_teste["texto_sem_stop_words"] = df_teste["texto_tokenizado"].apply(remove_stop_words)

df_teste[["texto_tokenizado", "texto_sem_stop_words"]].head()

,texto_tokenizado,texto_sem_stop_words
3,"[recebi, bem, antes, do, prazo, estipulado, .]","[recebi, bem, antes, prazo, estipulado]"
4,"[parabéns, lojas, lannister, adorei, comprar, ...","[parabéns, lojas, lannister, adorei, comprar, ..."
9,"[aparelho, eficiente, ., no, site, a, marca, d...","[aparelho, eficiente, site, marca, aparelho, i..."
12,"[mas, um, pouco, ,, travando, ..., pelo, valor...","[pouco, travando, valor, ta, boa]"
15,"[vendedor, confiável, ,, produto, ok, e, entre...","[vendedor, confiável, produto, ok, entrega, an..."


## Stemming - Reduzindo palavras ao seu radical

In [15]:
from nltk.stem import RSLPStemmer
nltk.download("rslp", quiet=True)

True

In [20]:
stemmer = RSLPStemmer()

def stemmerizacao(tokens):
    resultado = []

    for token in tokens:
        resultado.append(stemmer.stem(token))
    return resultado

In [21]:
df_teste["texto_stemming"] = df_teste["texto_sem_stop_words"].apply(stemmerizacao)

df_teste[["texto_sem_stop_words", "texto_stemming"]].head()

,texto_sem_stop_words,texto_stemming
3,"[recebi, bem, antes, prazo, estipulado]","[receb, bem, ant, praz, estipul]"
4,"[parabéns, lojas, lannister, adorei, comprar, ...","[parabém, loj, lannist, ador, compr, internet,..."
9,"[aparelho, eficiente, site, marca, aparelho, i...","[aparelh, efici, sit, marc, aparelh, impress, ..."
12,"[pouco, travando, valor, ta, boa]","[pouc, trav, val, ta, boa]"
15,"[vendedor, confiável, produto, ok, entrega, an...","[vend, confi, produt, ok, entreg, ant, praz]"


## 2. Pipeline Completo

In [22]:
def pipeline_tratamento(texto):

    texto_limpo = limpar_texto(texto)
    texto_tokenizado = tokenizacao(texto_limpo)
    texto_sem_stop_words = remove_stop_words(texto_tokenizado)
    texto_stemmer = stemmerizacao(texto_sem_stop_words)

    return " ".join(texto_stemmer)

In [24]:
dados["review"] = dados["review_comment_message"].apply(pipeline_tratamento)

dados[["review_comment_message", "review"]].head()

,review_comment_message,review
3,Recebi bem antes do prazo estipulado.,receb bem ant praz estipul
4,Parabéns lojas lannister adorei comprar pela I...,parabém loj lannist ador compr internet segur ...
9,aparelho eficiente. no site a marca do aparelh...,aparelh efici sit marc aparelh impress desinfe...
12,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",pouc trav val ta boa
15,"Vendedor confiável, produto ok e entrega antes...",vend confi produt ok entreg ant praz
